In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime
from functools import partial

# local imports
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.Constants import CTE as CTE

from makedf.mcstat import get_MCstat_unc

from analysis_village.cc1pi.var_configs import *

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result

extra_string = "_rate_no_bkg_substract"
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/syst/FullSyst" + extra_string

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Plot Syts

In [ ]:
var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]

In [ ]:
file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices" + extra_string

'''
flux_syst = np.load(file_dir + "/flux_syst_dict.npz")
g4_syst = np.load(file_dir + "/g4_syst_dict.npz")
genie_syst = np.load(file_dir + "/genie_syst_dict_xsec.npz")
'''
flux_syst = np.load(file_dir + "/extended_flux_syst_dict_rate_ar23p.npz")
g4_syst = np.load(file_dir + "/extended_g4_syst_dict_rate_ar23p.npz")
genie_syst = np.load(file_dir + "/extended_genie_syst_dict_xsec_ar23p.npz")
#GiBUU_syst = np.load(file_dir + "/GiBUU_syst_dict.npz")

file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices" 
mcstat_syst = np.load(file_dir + "/mcstat_syst_dict_ar23p.npz")
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")
detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz")

# flat uncertainties
pot_frac_unc = 0.02
ntargets_frac_unc = 0.01

In [ ]:
for var_config in var_configs:
    fig, ax = plt.subplots()
    
    frac_uncert_total = np.zeros(len(var_config.bin_centers))
    systs      = [mcstat_syst, flux_syst, g4_syst, genie_syst, cosmics_syst, detvar_syst]
    syst_names = ["MCStat", "Flux", "G4", "Genie", "Cosmic", "Detector"]
    #systs      = [mcstat_syst]
    #syst_names = ["MCStat"]
    print("-----")
    print("-----")
    print(genie_syst[var_config.var_save_name])
    print("-----")
    print("-----")
    # flat systs
    flat_systs = [pot_frac_unc, ntargets_frac_unc]
    flat_syst_names = ["POT", "Ntargets"]
    
  # First group (should appear on top)
    for syst_name, syst in zip(syst_names, systs):
        syst_uncert = np.sqrt(np.diag(syst[var_config.var_save_name]))
        frac_uncert_total += syst_uncert ** 2
        print(syst_name, syst_uncert*100)
        plt.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=syst_uncert * 1e2,
            histtype="step",
            linewidth=2,
            label=syst_name,
            zorder=3
        )
    
    # Second group (should appear underneath)
    for syst_name, syst in zip(flat_syst_names, flat_systs):
        syst_uncert = syst * np.ones(len(var_config.bin_centers))
        frac_uncert_total += syst_uncert ** 2
        print(syst_name, syst_uncert*100)
    
        plt.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=syst_uncert * 1e2,
            histtype="step",
            linewidth=2,
            label=syst_name,
            zorder=1
        )
    
    frac_uncert_total = np.sqrt(frac_uncert_total)
    print("Total", frac_uncert_total*100)
    plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_uncert_total * 1e2,    histtype="step", linewidth=2, color="k",  label="Total")
    
    plt.xlim(var_config.bins[0], var_config.bins[-1])
    plt.ylim(0, max(frac_uncert_total*1e2) * 1.4)
    
    plt.xlabel(var_config.var_labels[1])
    plt.ylabel("Uncertainty [%]")
    plt.legend(fontsize=11, ncol=3, loc="upper center")
    
    plt.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
    plt.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
    plt.minorticks_on()

    f_name = f"full_syst_{var_config.var_save_name}.pdf" 
    save_full_path = os.path.join(save_fig_dir, f_name)
    fig.savefig(save_full_path, format='pdf', bbox_inches='tight')

In [ ]:
from analysis_village.cc1pi.CutMasks.MaskUtils import *

pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100, reprocess_df = False, reprocess_truth = False)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']


keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
#Load data
keys2load = ["cc1pi_good", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_fixdev_bnblight_quality_cut.df", keys2load, 100)
data_evt_df = data_df['cc1pi_good']
data_hdr_df = data_df['hdr']
del data_df
gc.collect()


# BNB data
print("data_tot_pot: %.3e" %(data_tot_pot))
print("data tot gates : %.3e" %(data_gates))
#data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))

mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))
print(len(mc_bnb_evt_df))

#Do truth matchign
mc_evt_df = mc_bnb_evt_df
if "ar23p" in bnb_path:
    print("NICE")
    #mc_evt_df = perform_truth_matching_low_memmory(mc_bnb_evt_df, mc_bnb_nu_df)
else:
    mc_evt_df = perform_truth_matching(mc_bnb_evt_df, mc_bnb_nu_df)   
'''

if "ar23p" not in bnb_path:
    mc_bnb_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_nu_df))
else:
    new_columns = []
    for c in mc_bnb_nu_df.columns:
        new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
    mc_bnb_nu_df.columns = pd.MultiIndex.from_tuples(new_columns)
    mc_bnb_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_nu_df))
'''  

mc_cumulative_masks = build_event_cumulative_masks(mc_evt_df, sideband = "")
mc_evt_df = mc_evt_df[mc_cumulative_masks["energy"]]

mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first())

In [ ]:
from pyanalib.covariance import *

systs      = [flux_syst, g4_syst, genie_syst,  detvar_syst, cosmics_syst]
syst_names = ["Flux", "G4", "Genie", "Detector", "Cosmic"]

flat_systs = [pot_frac_unc, ntargets_frac_unc]
flat_syst_names = ["POT", "Ntargets"]

save_fig = True

for var_config in var_configs:
    total_cov_frac = mcstat_syst[var_config.var_save_name].copy()
    for name, syst_dict in zip(syst_names, systs):
        matrix = syst_dict[var_config.var_save_name]
        total_cov_frac += matrix # Matrix addition preserves correlations

    for syst_name, syst in zip(flat_syst_names, flat_systs):
        # 1. Create the vector of uncertainties (syst can be a scalar or an array)
        syst_vector = syst * np.ones(len(var_config.bin_centers))
        syst_matrix_sq = np.diag(syst_vector**2)
        total_cov_frac += syst_matrix_sq

    ret_nom = signal_hists(mc_evt_df, mc_bnb_nu_df, var_config, return_data=True, plot=False)
    total_cov = cov_from_fraccov(total_cov_frac, ret_nom["nevts_sel_reco"])
    total_corr = corr_from_fraccov(total_cov)

    f_name = f"full_syst_{var_config.var_save_name}_frac_cov" 
    save_full_path = os.path.join(save_fig_dir, f_name)
       
    plot_heatmap(total_cov_frac, 
            var_config.bins, 
            plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
            save_fig=save_fig, save_name=save_full_path)

    f_name = f"full_syst_{var_config.var_save_name}_cov" 
    save_full_path = os.path.join(save_fig_dir, f_name)
    plot_heatmap(total_cov, 
            var_config.bins, 
            plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
            save_fig=save_fig, save_name=save_full_path)
        
    f_name = f"full_syst_{var_config.var_save_name}_corr" 
    save_full_path = os.path.join(save_fig_dir, f_name)
    plot_heatmap(total_corr, 
            var_config.bins, 
            plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
            save_fig=save_fig, save_name=save_full_path)